In [ ]:
import pandas as pd
import numpy as np
import os

print("=" *60)
print("DATA VALIDATION AND CLEANING")
print("=" *60)


#1.loading datasets and getting baseline counts
prices_path = "data/raw/prices/raw_maize_market_prices.csv"
weather_path = "data/raw/weather/raw_county_weather.csv"

df_prices = pd.read_csv(prices_path)
df_weather = pd.read_csv(weather_path)

rows_received_prices = len(df_prices)
rows_received_weather = len(df_weather)

#2.view the 5 rows to see the columns and values look like
print("\n---First 5 Rows of Raw Prices Data ---")
display(df_prices.head())

#3.view data types and  non null counts (to see where blanks exist)
print("\n---Data Information and Null counts---")
df_prices.info()
#4.check the exact missing values dynamically
print("\n---Exact Missing Values per column ---")
print(df_prices.isna().sum())
#5.check unique values in categorical columns to see inconsistencies(eg messycounties or units)
print("\n---Unique Units in Raw Data (Spotting inconsistencies)---")
print(df_prices['unit'].unique())

#PART 2:CLEANING
print("=" * 60)
print("RUNNING DATA CLEANING")
print("=" * 60)

rows_received_prices = len(df_prices)
rows_received_weather = len(df_weather)

#1.track initial missing values dynamically from our inspection
initial_missing_prices = df_prices['wholesale_price'].isna().sum()
initial_missing_retail = df_prices['retail_price_per_kg'].isna().sum()
initial_missing_supply = df_prices['supply_volume_tonnes'].isna().sum()

#2.handling duplicate records
duplicates_count = df_prices.duplicated().sum()
df_prices_dedup = df_prices.drop_duplicates().copy

#3.cleaning county and market names
df_prices_dedup['county_name'] = df_prices_dedup['county_name'].astype(str).str.strip().str.title()
df_prices_dedup['market_name'] = df_prices_dedup['market_name'].astype(str).strip().str.title()

#4.cleaning currency formats and malformed numbers
def clean_currency_to_float(val):
    if pd.isna(val):
        return np.nan

    if isinstance(val,(int,float)):
        return float(val)
    cleaned = str(val).upper().replace("KES","").replace(",","").strip()
    try:
        return float(cleaned)
    except ValueError:
        return np.nan
    
df_prices_dedup['wholesale_price_cleaned'] = df_prices_dedup['wholesale_price'].apply(clean_currency_to_float)
malformed_prices_count = df_prices_dedup['wholesale_price_cleaned'].isna().sum()

    
    